In [1]:
import ipywidgets as widgets
from diffcalc.hkl.calc import HklCalculation
from diffcalc.ub.calc import UBCalculation, Crystal
from diffcalc.hkl.constraints import Constraints
from xrayutilities.utilities import en2lam
import pandas as pd

res_str = ''


def calc_angles(
    lat_name = 'custom lattice',
    uc_a = 7.27, 
    uc_b = 5.0, 
    uc_c = 5.55, 
    uc_alpha = 90, 
    uc_beta = 96.8, 
    uc_gamma = 90,
    surface_normal = '(0,0,1)',
    orientations = [''],
    h=0,k=1,l=3,
    delta = 0,
    delta_fixed = False,
    gam = 0,
    gam_fixed = False,
    qaz = 0,
    qaz_fixed = False,
    naz = 0,
    naz_fixed = False,
    alpha = 0,
    alpha_fixed = False,
    beta = 0,
    beta_fixed = False,
    psi = 0,
    psi_fixed = False,
    a_eq_b = 0,
    a_eq_b_fixed = False,
    mu = 0,
    mu_fixed = False,
    eta = 0,
    eta_fixed = False,
    chi = 0,
    chi_fixed = True,
    phi = 0,
    phi_fixed = False,
    mu_is_gam = 0,
    mu_is_gam_fixed = False,
    energy = 8000,
    ):
    surf_dir = (0,0,1)
    ubcalc = UBCalculation('you')
    ubcalc.set_lattice('V2O3',a = uc_a,b = uc_b,c = uc_c,alpha = uc_alpha,beta = uc_beta,gamma = uc_gamma)
    # sample normal 
    
    # setting c in direction to the phi rotation axis
    for to in orientations:
        ubcalc.add_orientation(
            eval(to.split('>')[0].strip()),
            eval(to.split('>')[1].strip()),
            None,None)
            
                 
    # ubcalc.add_orientation((1,0,0),(0,0,1),None,"c-dir") 
    # ubcalc.add_orientation((1,1,1),(0,-1,0),None,"surf direction")

    res_str = ', '.join(orientations)
    ubcalc.n_hkl = eval(surface_normal)
    
    ubcalc.calc_ub()


    lam = en2lam(energy)
    pd.set_option('display.float_format', lambda x: '%.3f' % x)
    constraints = {}
    
    cnames = ['delta','nu','qaz','naz','alpha','beta','psi','a_eq_b','mu','eta','chi','phi','mu_is_gam']
    cvalues = [delta,gam,qaz,naz,alpha,beta,psi,a_eq_b,mu,eta,chi,phi,mu_is_gam]
    cfixed = [delta_fixed, gam_fixed, qaz_fixed, naz_fixed, alpha_fixed, beta_fixed, psi_fixed, a_eq_b_fixed, mu_fixed, eta_fixed,chi_fixed,phi_fixed,mu_is_gam_fixed]
    try:
        cons = Constraints({tn:tv for tn,tf,tv in zip(cnames,cfixed,cvalues) if tf})
        hklcalc = HklCalculation(ubcalc, cons)
        result = hklcalc.get_position(h,k,l,lam)
    except Exception as e:
        print(str(e))
        return
    result = pd.concat([pd.DataFrame.from_dict({**tres[0].asdict,**tres[1]},orient='index', columns=[f'sol. {n}']) for n,tres in enumerate(result)],axis=1)
    result.rename(index={"nu": "gamma"},inplace=True)
    # print(result)
    # return result.T.loc[(result.T['gamma']>=0)]# & (result.T['mu']>90) & (result.T['mu']<90+180)]
    display(result)

ucb = widgets.HBox()
ucb.children = [
    widgets.FloatText(value=3),
    widgets.FloatText(value=3),
    widgets.FloatText(value=3),
]

ucparw = '150px'
angparw = '150px'
hklparw = '150px'
w = widgets.interactive(calc_angles,
         uc_a=widgets.FloatText(description='a$_{uc}$',value=5.43,layout = widgets.Layout(max_width=ucparw)),
         uc_b=widgets.FloatText(description='b$_{uc}$',value=5.43,layout = widgets.Layout(max_width=ucparw)),
         uc_c=widgets.FloatText(description='c$_{uc}$',value=5.43,layout = widgets.Layout(max_width=ucparw)),
         uc_alpha=widgets.FloatText(description='$\\alpha_{uc}$',value=90,layout = widgets.Layout(max_width=ucparw)),
         uc_beta=widgets.FloatText(description='$\\beta_{uc}$',value=90,layout = widgets.Layout(max_width=ucparw)),
         uc_gamma=widgets.FloatText(description='$\\gamma_{uc}$',value=90,layout = widgets.Layout(max_width=ucparw)),
         # so_dir = widgets.TagsInput(),
                        # uc_a=widgets.FloatText(value=3),
         # uc_b=widgets.FloatText(value=3),
         # uc_c=widgets.FloatText(value=3),
         h = widgets.IntText(value=1,layout = widgets.Layout(max_width=hklparw,positioning='right')),
         k = widgets.IntText(value=1,layout = widgets.Layout(max_width=hklparw,positioning='right')),
         l = widgets.IntText(value=1,layout = widgets.Layout(max_width=hklparw,positioning='right')),
         surface_normal = widgets.Text(value='(1,1,1)'),
         orientations = widgets.TagsInput(value=['(1,1,1) > (0,0,1)', '(1,1,-1) > (1,0,0)'],description='orientations'),
         delta = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         delta_fixed = False,
         gam = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         gam_fixed = False,
         qaz = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         qaz_fixed = False,
         naz = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         naz_fixed = False,
         alpha = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         alpha_fixed = False,
         beta = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         beta_fixed = False,
         psi = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         psi_fixed = False,
         a_eq_b = widgets.Checkbox(value=0,layout = widgets.Layout(max_width=angparw)),
         a_eq_b_fixed = False,
         mu = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         mu_fixed = False,
         eta = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         eta_fixed = True,
         chi = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         chi_fixed = True,
         phi = widgets.FloatText(value=0,layout = widgets.Layout(max_width=angparw)),
         phi_fixed = True,
         mu_is_gam = widgets.Checkbox(value=0,layout = widgets.Layout(max_width=angparw)),
         mu_is_gam_fixed = False,
         energy = widgets.FloatSlider(value=8000,min=1000,max=20000),
         # alpha=FloatSlider(value=16,min=0,max=90,disabled=False),
        );

ctr_uc = widgets.HBox(w.children[1:7], layout = widgets.Layout(flex_flow='row wrap'))
ctr_or = widgets.VBox(w.children[7:9],layout = widgets.Layout(positioning='right'))
ctr_hkl = widgets.HBox(w.children[9:12])#, layout = widgets.Layout(flex_flow='row wrap'))
ctmp = w.children[12:38]
res = list(zip(ctmp[::2], ctmp[1::2]))
ctr_ang = widgets.VBox([widgets.HBox(list(tg)) for tg in zip(ctmp[::2], ctmp[1::2])])
ctr_tmp2 = w.children[-2]
controls = widgets.VBox([ctr_uc,ctr_or,ctr_hkl,ctr_ang, ctr_tmp2],layout = widgets.Layout(max_width='50%'))
output = w.children[-1]
display(widgets.HBox([controls, output]))
# display(w)





from ipywidgets import HTML
from IPython.display import display

import base64



#FILE


filename = 'res.txt'
def get_pars():
    b64 = base64.b64encode(res_str.encode())
    payload = b64.decode()
    return payload

#BUTTONS
html_buttons = '''<html>
<head>
<meta name="viewport" content="width=device-width, initial-scale=1">
</head>
<body>
<a download="{filename}" href="data:text/csv;base64,{payload}" download>
<button class="p-Widget jupyter-widgets jupyter-button widget-button mod-warning">Download File</button>
</a>
</body>
</html>
'''

html_button = html_buttons.format(payload=get_pars(),filename=filename)
display(HTML(html_button))

HTML(value='<html>\n<head>\n<meta name="viewport" content="width=device-width, initial-scale=1">\n</head>\n<bo…